# K-means, Expectation-Maximisation and Boltzmann Machines

Prepared for Fall 2026. Reference: the notes for Lecture 8 (K-means, Expectation-Maximisation and Boltzmann Machines: learning probability distributions with hidden variables).

If you find a bug in this notebook, please email the TAs or tell us at office hours.

**Before you start.** Run the notebook from the homework folder, so that the folder `data` is next to it (this notebook uses the photographs in `data/edge_detection`). Every figure and number that a question asks for must appear in your PDF: when a question asks you to vary a parameter, copy the relevant cell into the code cell below the question (or write a loop) instead of editing and re-running the cell above, which would overwrite the earlier figure.

---

All three models of this notebook are learned from data in which some variables are hidden:

- **k-means** splits unlabelled data into $k$ classes; the class of every datapoint is hidden;
- **EM for a mixture of Gaussians** (soft k-means) replaces the hard class assignments by probabilities;
- a **Boltzmann machine** is a Gibbs distribution over binary neurons, some of which are hidden.

## Utilities

In [ ]:
from __future__ import annotations
import itertools
import os

import numpy as np
import numpy.typing as npt
import matplotlib.pyplot as plt
from PIL import Image

%matplotlib inline

if not os.path.isdir("data/edge_detection"):
    raise FileNotFoundError("The folder data/edge_detection was not found. Start Jupyter in the homework folder, "
                            "so that the folder data is next to this notebook.")


def make_blobs(centres, n_per_class: int = 30, sigma: float = 0.5, seed: int = 0):
    """Datapoints drawn from isotropic Gaussians with standard deviation sigma around the given centres."""
    rng = np.random.default_rng(seed)
    X = np.concatenate([rng.normal(c, sigma, size=(n_per_class, 2)) for c in centres])
    y = np.repeat(np.arange(len(centres)), n_per_class)
    return X, y


def kmeans_energy(X, means, assignment) -> float:
    """E = sum over datapoints of the squared distance to the mean of their class."""
    return float(np.sum((X - means[assignment]) ** 2))


def kmeans_plus_plus(X, k: int, rng) -> npt.NDArray:
    """k-means++ seeding: each new mean is a datapoint drawn with probability proportional to its
    squared distance to the nearest mean already chosen."""
    means = [X[rng.integers(len(X))]]
    for _ in range(k - 1):
        d2 = np.min(((X[:, None, :] - np.array(means)[None]) ** 2).sum(-1), axis=1)
        means.append(X[rng.choice(len(X), p=d2 / d2.sum())])
    return np.array(means)


def kmeans(X, k: int, init: str = "random partition", seed: int = 0, max_iter: int = 100):
    """K-means. Returns the means, the assignment and the energy after every half-step.

    init = "random partition": every datapoint is assigned to a random class and the first means
                               are the class averages (step 1 of the lecture)
    init = "k-means++":        the first means are chosen by k-means++
    """
    rng = np.random.default_rng(seed)
    if init == "random partition":
        assignment = rng.integers(k, size=len(X))
        means = np.array([X[assignment == a].mean(0) if np.any(assignment == a) else X[rng.integers(len(X))] for a in range(k)])
    elif init == "k-means++":
        means = kmeans_plus_plus(X, k, rng)
    else:
        raise ValueError(init)
    energies = []
    for _ in range(max_iter):
        # step 3: assign every datapoint to the nearest mean (ties go to the smaller index)
        d2 = ((X[:, None, :] - means[None]) ** 2).sum(-1)
        assignment = d2.argmin(1)
        energies.append(kmeans_energy(X, means, assignment))
        # step 2: the mean of every class is the average of its datapoints (an empty class is re-initialised
        # at the datapoint furthest from its own mean)
        new_means = np.array([X[assignment == a].mean(0) if np.any(assignment == a) else X[d2.min(1).argmax()] for a in range(k)])
        energies.append(kmeans_energy(X, new_means, assignment))
        if np.allclose(new_means, means):
            break
        means = new_means
    return means, assignment, np.array(energies)


def plot_partition(X, means, assignment, ax, title: str = ""):
    """Datapoints coloured by class, the means as stars, and the region closest to each mean shaded."""
    colours = plt.cm.tab10(np.arange(len(means)) % 10)
    x0, x1 = X[:, 0].min() - 1, X[:, 0].max() + 1
    y0, y1 = X[:, 1].min() - 1, X[:, 1].max() + 1
    gx, gy = np.meshgrid(np.linspace(x0, x1, 300), np.linspace(y0, y1, 300))
    grid = np.stack([gx.ravel(), gy.ravel()], 1)
    region = ((grid[:, None, :] - means[None]) ** 2).sum(-1).argmin(1).reshape(gx.shape)
    ax.imshow(colours[region][..., :3] * 0.25 + 0.75, extent=(x0, x1, y0, y1), origin="lower")
    ax.scatter(X[:, 0], X[:, 1], c=colours[assignment], s=12)
    ax.scatter(means[:, 0], means[:, 1], c=colours[: len(means)], marker="*", s=300, edgecolor="k")
    ax.set_title(title)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])

## K-means

K-means minimises the energy $E(\{V\}, \{m\}) = \sum_{i}\sum_{a} V_{ia}\, |x_i - m_a|^2$ over the assignments $V_{ia} \in \{0, 1\}$ and the means $m_a$. In the numbering of Lecture 8 (pp. 41–42), step 1 initialises the algorithm (a random partition of the data, or k-means++, which supplies means and so starts at step 3); step 2 sets each mean to the average of its class; step 3 assigns every datapoint to the nearest mean; and step 4 repeats steps 2 and 3 until the assignment stops changing. Steps 2 and 3 are exact minimisations of $E$, over the means and over the assignments. The data below come from six Gaussian clusters on a $2 \times 3$ grid.

In [ ]:
centres = [(0, 0), (3, 0), (6, 0), (0, 3), (3, 3), (6, 3)]
X, _ = make_blobs(centres, n_per_class=30, sigma=0.5, seed=1)
k = 6

means, assignment, energies = kmeans(X, k, init="random partition", seed=2)   # try other seeds for Question 10.1
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(energies, marker="o", ms=4)
axes[0].set_xlabel("half-step")
axes[0].set_ylabel("energy E")
axes[0].set_title("one run from a random partition")
plot_partition(X, means, assignment, axes[1], f"final partition, E = {energies[-1]:.1f}")
fig.tight_layout()
plt.show()

In [ ]:
n_runs = 50
final = {init: np.array([kmeans(X, k, init=init, seed=s)[2][-1] for s in range(n_runs)]) for init in ("random partition", "k-means++")}
best = min(e.min() for e in final.values())
for init, e in final.items():
    print(f"{init:17s}: lowest energy {e.min():6.1f}, fraction of runs within 1 of the lowest: {np.mean(e < best + 1):.2f}")

plt.figure(figsize=(6, 3.5))
plt.hist(list(final.values()), bins=30, label=list(final.keys()))
plt.xlabel("final energy E")
plt.ylabel("number of runs")
plt.legend()
plt.show()

### Question 10.1 (3 points)

- Plot the final partition of a run that reaches the lowest energy and of a run that stops at a higher energy, with their energies (try a few values of `seed`, with `kmeans` and `plot_partition` in the code cell below). **(1 point)**
- Explain, using the two steps of the algorithm, why a run that has stopped at a higher energy cannot improve any further. **(1 point)**
- Report the fraction of runs that reach the lowest energy (within 1 of it) with random partitions and with k-means++. Why does k-means++ help? **(1 point)**

In [ ]:
# Your code for Question 10.1

**Your answers for Question 10.1:**

## EM for a mixture of Gaussians

The mixture model says that each datapoint was generated by first choosing one of $k$ classes with equal probability and then drawing from the Gaussian of that class, with mean $m_a$ and covariance $\sigma^2 I$. With $\sigma$ fixed, EM alternates

- **E-step:** $\;P(\omega_a \mid x_j) = \dfrac{\exp\{-|x_j - m_a|^2 / 2\sigma^2\}}{\sum_b \exp\{-|x_j - m_b|^2 / 2\sigma^2\}}$, the soft assignment of datapoint $x_j$ to class $a$;
- **M-step:** $\;m_a = \dfrac{\sum_j x_j\, P(\omega_a \mid x_j)}{\sum_j P(\omega_a \mid x_j)}$, the weighted mean.

The data are 120 points from three Gaussians with $\sigma = 0.5$, as in the lecture notes.

In [ ]:
def log_likelihood(X, means, sigma: float) -> float:
    """log P(X | means) for the equal-weight mixture of isotropic Gaussians in two dimensions."""
    d2 = ((X[:, None, :] - means[None]) ** 2).sum(-1)
    log_components = -d2 / (2 * sigma**2) - np.log(2 * np.pi * sigma**2) - np.log(len(means))
    top = log_components.max(1, keepdims=True)
    return float(np.sum(top[:, 0] + np.log(np.exp(log_components - top).sum(1))))


def em_mixture(X, means0, sigma: float, n_iter: int = 30):
    """EM for the equal-weight mixture of isotropic Gaussians with known sigma.

    Returns the means, the responsibilities P(class | datapoint) for the returned means, and the
    log-likelihood before every iteration and after the last one.
    """
    def e_step(means):
        d2 = ((X[:, None, :] - means[None]) ** 2).sum(-1)
        logits = -d2 / (2 * sigma**2)
        r = np.exp(logits - logits.max(1, keepdims=True))
        return r / r.sum(1, keepdims=True)

    means = np.array(means0, dtype=float)
    history = []
    for _ in range(n_iter):
        history.append(log_likelihood(X, means, sigma))
        r = e_step(means)                                                   # E-step
        soft_counts = r.sum(0)
        new_means = (r.T @ X) / np.maximum(soft_counts, 1e-12)[:, None]     # M-step
        means = np.where(soft_counts[:, None] > 1e-12, new_means, means)    # a class with no responsibility keeps its mean
    history.append(log_likelihood(X, means, sigma))
    return means, e_step(means), np.array(history)


def plot_soft(X, means, r, ax, title: str = ""):
    """Each datapoint is coloured by blending the three class colours with its responsibilities."""
    colours = np.array([[0.84, 0.15, 0.16], [0.17, 0.63, 0.17], [0.12, 0.47, 0.71]])
    ax.scatter(X[:, 0], X[:, 1], c=np.clip(r @ colours, 0, 1), s=14)
    ax.scatter(means[:, 0], means[:, 1], c=colours, marker="*", s=300, edgecolor="k")
    ax.set_title(title)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])


X3, _ = make_blobs([(0, 0), (2.6, 0.7), (1.1, 2.4)], n_per_class=40, sigma=0.5, seed=3)
means0 = X3.mean(0) + np.array([[-0.3, -0.2], [0.3, -0.1], [0.0, 0.3]])     # started near the centre of the data

means, r, history = em_mixture(X3, means0, sigma=0.5)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(history, marker="o", ms=4)
axes[0].set_xlabel("iteration")
axes[0].set_ylabel("log-likelihood")
plot_soft(X3, means, r, axes[1], r"soft assignments after EM, $\sigma = 0.5$")
fig.tight_layout()
plt.show()
print(f"log-likelihood: {history[0]:.1f} before the first iteration, {history[-1]:.1f} after the last")
print("log-likelihood never decreases:", bool(np.all(np.diff(history) >= -1e-9)))
print("final means:\n", np.round(means, 2))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, sigma in zip(axes[:3], (1.5, 0.5, 0.1)):
    m, r_sigma, _ = em_mixture(X3, means0, sigma=sigma, n_iter=50)
    coincide = np.ptp(m, axis=0).max() < 0.05
    plot_soft(X3, m, r_sigma, ax, rf"EM, $\sigma = {sigma}$" + (" (the three means coincide)" if coincide else ""))
    print(f"sigma = {sigma}: average of the largest responsibility of each point {r_sigma.max(1).mean():.3f}; means {np.round(m, 2).tolist()}")

# k-means from the same starting means, for comparison (one-hot assignments drawn with the same colours)
m = means0.copy()
for _ in range(50):
    a = ((X3[:, None, :] - m[None]) ** 2).sum(-1).argmin(1)
    m = np.array([X3[a == c].mean(0) if np.any(a == c) else m[c] for c in range(3)])   # an empty class keeps its mean
plot_soft(X3, m, np.eye(3)[a], axes[3], "k-means")
fig.tight_layout()
plt.show()

### Question 10.2 (3 points)

- Plot the log-likelihood over the EM iterations. Does it ever decrease? Which property of EM guarantees this? **(1 point)**
- Show the soft assignments for $\sigma = 1.5$, $0.5$ and $0.1$. Describe what happens to the assignments as $\sigma$ gets smaller, and explain how EM for this mixture is related to k-means. What goes wrong when $\sigma$ is much larger than the spread of each cluster? **(2 points)**

In [ ]:
# Your code for Question 10.2

**Your answers for Question 10.2:**

## Spherical k-means on image patches

The brightness of an image patch depends on the lighting as much as on the pattern in the patch. Subtracting the mean of the patch removes the background level, and dividing by its length removes the contrast, so every normalised patch is a unit vector. Spherical k-means assigns each patch to the mean with the largest dot product and sets each mean to the normalised sum of its patches: this is the winner-take-all network of the lecture, and the zero-variance limit of a mixture of von Mises-Fisher distributions.

The patches are $8 \times 8$ windows taken at random from the seven photographs of the edge detection notebook.

In [ ]:
def sample_patches(n: int = 20000, size: int = 8, seed: int = 0) -> npt.NDArray:
    """Random size x size grey-level patches from the seven edge-detection photographs, flattened."""
    rng = np.random.default_rng(seed)
    images = [np.array(Image.open(f"data/edge_detection/images/{i}.jpg").convert("L")).astype(float) / 255 for i in range(7)]
    patches = []
    for _ in range(n):
        im = images[rng.integers(len(images))]
        y, x = rng.integers(im.shape[0] - size), rng.integers(im.shape[1] - size)
        patches.append(im[y:y + size, x:x + size].ravel())
    return np.array(patches)


def normalise(patches, min_norm: float = 0.05):
    """Subtract the mean of every patch and scale it to unit length; patches with almost no contrast are dropped."""
    centred = patches - patches.mean(1, keepdims=True)
    norm = np.linalg.norm(centred, axis=1)
    keep = norm > min_norm
    return centred[keep] / norm[keep, None]


def spherical_kmeans(X, k: int, n_iter: int = 30, seed: int = 0):
    """Spherical k-means on unit vectors: winner-take-all by the largest dot product, means renormalised."""
    rng = np.random.default_rng(seed)
    means = X[rng.choice(len(X), k, replace=False)].copy()
    for _ in range(n_iter):
        assignment = (X @ means.T).argmax(1)
        for a in range(k):
            total = X[assignment == a].sum(0)
            if np.linalg.norm(total) > 0:
                means[a] = total / np.linalg.norm(total)
    assignment = (X @ means.T).argmax(1)          # the assignment for the final means
    return means, assignment


def show_patches(means, counts=None, title: str = "", vmin=None, vmax=None):
    order = np.argsort(-counts) if counts is not None else np.arange(len(means))
    fig, axes = plt.subplots(2, len(means) // 2, figsize=(len(means) * 0.8, 2.2))
    for ax, a in zip(axes.flat, order):
        ax.imshow(means[a].reshape(8, 8), cmap="gray", vmin=vmin, vmax=vmax)
        if counts is not None:
            ax.set_title(str(counts[a]), fontsize=8)
        ax.axis("off")
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()


raw = sample_patches()
unit = normalise(raw)
print(f"{len(unit)} of {len(raw)} patches kept after normalisation")

means, assignment = spherical_kmeans(unit, k=16)
show_patches(means, np.bincount(assignment, minlength=16), "spherical k-means on normalised patches (number of patches above each mean)")

# For comparison: plain k-means on the raw patches (no normalisation)
raw_means, raw_assignment, _ = kmeans(raw[:8000], 16, init="k-means++", seed=0)
show_patches(raw_means, np.bincount(raw_assignment, minlength=16), "plain k-means on raw patches", vmin=0, vmax=1)

### Question 10.3 (2 points)

- Show the 16 means found by spherical k-means. What do they look like, and how are they related to the filters of Lecture 4 (derivatives of Gaussians and Gabor filters)? **(1 point)**
- Compare them with the means of plain k-means on the raw patches. What do those means capture, and why is contrast normalisation needed to find the structure in the patches? **(1 point)**

In [ ]:
# Your code for Question 10.3

**Your answers for Question 10.3:**

## A Boltzmann machine that learns bars

The machine has 9 observed neurons, drawn as the pixels of a $3 \times 3$ image, and 2 hidden neurons. Every pair of neurons is joined by a symmetric weight $\omega_{ij}$, and the machine defines the Gibbs distribution

$$E(\vec S) = -\frac{1}{2}\sum_{i,j}\omega_{ij} S_i S_j, \qquad P(\vec S) = \frac{1}{Z}\exp\{-E(\vec S)/T\}.$$

The training data are the six images that contain exactly one horizontal or one vertical bar. Learning changes every weight by

$$\Delta\omega_{ij} = \frac{\zeta}{T}\Big\{\langle S_i S_j\rangle_{\text{clamped}} - \langle S_i S_j\rangle_{\text{free}}\Big\}.$$

The machine is small ($2^{11} = 2048$ states), so both correlations are computed exactly by enumerating the states, and the KL divergence between the data and the machine's distribution over the observed neurons can be tracked. A larger machine would need Gibbs sampling for both.

In [ ]:
def bars(n: int = 3) -> npt.NDArray:
    """All n x n images with exactly one horizontal or one vertical bar, flattened."""
    images = []
    for r in range(n):
        im = np.zeros((n, n)); im[r, :] = 1; images.append(im.ravel())
    for c in range(n):
        im = np.zeros((n, n)); im[:, c] = 1; images.append(im.ravel())
    return np.array(images)


def all_states(n_units: int) -> npt.NDArray:
    return np.array(list(itertools.product([0, 1], repeat=n_units)), dtype=float)


def log_boltzmann_probabilities(states, W, T: float = 1.0):
    """log P(S) for each of the given states, computed in the log domain so that it never underflows."""
    energies = -0.5 * np.einsum("si,ij,sj->s", states, W, states)
    logits = -energies / T
    top = logits.max()
    return logits - top - np.log(np.exp(logits - top).sum())


def boltzmann_probabilities(states, W, T: float = 1.0):
    return np.exp(log_boltzmann_probabilities(states, W, T))


def learn_boltzmann(data, n_hidden: int = 2, T: float = 1.0, rate: float = 0.5, n_steps: int = 400,
                    free_phase: bool = True, seed: int = 0):
    """Boltzmann machine learning with exact clamped and free correlations.

    Returns the weights and the KL divergence between the data and the machine (over the observed
    neurons) before every step. With free_phase=False only the clamped (Hebbian) term is used.
    """
    rng = np.random.default_rng(seed)
    n_visible = data.shape[1]
    n_units = n_visible + n_hidden
    W = rng.normal(scale=0.1, size=(n_units, n_units))
    W = (W + W.T) / 2
    np.fill_diagonal(W, 0)
    states = all_states(n_units)
    hidden_states = all_states(n_hidden)
    codes = (states[:, :n_visible] @ 2 ** np.arange(n_visible)).astype(int)
    data_codes = (data @ 2 ** np.arange(n_visible)).astype(int)
    kl = []
    for step in range(n_steps + 1):
        log_p = log_boltzmann_probabilities(states, W, T)
        p = np.exp(log_p)
        # KL(data || machine) over the observed neurons: the average of log(1/N) - log P(v) over the N training images
        log_p_data = np.array([np.logaddexp.reduce(log_p[codes == c]) for c in data_codes])
        kl.append(float(np.mean(-np.log(len(data)) - log_p_data)))
        if step == n_steps:
            break
        clamped = np.zeros((n_units, n_units))
        for v in data:                               # observed neurons held at a training image
            s = np.hstack([np.tile(v, (len(hidden_states), 1)), hidden_states])
            pc = boltzmann_probabilities(s, W, T)
            clamped += (s * pc[:, None]).T @ s / len(data)
        free = (states * p[:, None]).T @ states if free_phase else 0.0
        dW = rate / T * (clamped - free)
        np.fill_diagonal(dW, 0)
        W += dW
    return W, np.array(kl)


def gibbs_chain(W, n_visible: int, n_updates: int = 6000, T: float = 1.0, every: int = 250, seed: int = 0):
    """Gibbs sampling, one randomly chosen neuron at a time; the observed neurons are recorded every `every` updates."""
    rng = np.random.default_rng(seed)
    s = (rng.random(len(W)) < 0.5).astype(float)
    snapshots = []
    for t in range(1, n_updates + 1):
        i = rng.integers(len(W))
        s[i] = float(rng.random() < 1 / (1 + np.exp(-np.clip((W[i] @ s) / T, -50, 50))))
        if t % every == 0:
            snapshots.append(s[:n_visible].copy())
    return np.array(snapshots)


def show_images(images, title: str = "", n: int = 3):
    fig, axes = plt.subplots(1, len(images), figsize=(0.7 * len(images), 1.1))
    for ax, im in zip(np.atleast_1d(axes), images):
        ax.imshow(im.reshape(n, n), cmap="Blues", vmin=0, vmax=1)
        ax.set_xticks([])
        ax.set_yticks([])
    fig.suptitle(title, fontsize=9)
    plt.show()


data = bars(3)
show_images(data, "the training data")

W, kl = learn_boltzmann(data, n_hidden=2)
plt.figure(figsize=(4.5, 3))
plt.plot(kl)
plt.xlabel("learning step")
plt.ylabel("KL(data || machine)")
plt.show()

states = all_states(len(W))
p = boltzmann_probabilities(states, W)
codes = (states[:, :9] @ 2 ** np.arange(9)).astype(int)
p_visible = np.bincount(codes, weights=p, minlength=2**9)
print(f"KL after learning: {kl[-1]:.3f}")
print(f"probability of the six bar images under the machine: {p_visible[(data @ 2 ** np.arange(9)).astype(int)].sum():.3f}")

snapshots = gibbs_chain(W, n_visible=9)
show_images(snapshots, "Gibbs samples of the trained machine, every 250 single-neuron updates")

In [ ]:
# Learning with the clamped (Hebbian) term only. 100 steps are enough to see what happens:
# without the free term nothing stops the weights from growing.
W_hebb, kl_hebb = learn_boltzmann(data, n_hidden=2, free_phase=False, n_steps=100)
print(f"KL after 100 Hebbian-only steps: {kl_hebb[-1]:.3f}")
show_images(gibbs_chain(W_hebb, n_visible=9, n_updates=2000, every=100), "Gibbs samples, Hebbian term only")

### Question 10.4 (2 points)

- Plot the KL divergence during learning and show the Gibbs samples of the trained machine. How often does the chain move from one bar to another, and why is this a problem for estimating the free correlations by sampling in a larger machine? **(1 point)**
- Train again with the clamped (Hebbian) term only. What does the machine generate now? Explain why the free (anti-Hebbian) term is needed: what does it measure, and what happens to the weights without it? **(1 point)**

In [ ]:
# Your code for Question 10.4

**Your answers for Question 10.4:**